# Домашнее задание №2 · Обучение без учителя

МФТИ · «Практическое машинное обучение» · максимум 100 баллов.

**Дедлайн: 6 ноября 2026 года, 23:59 МСК.** При опоздании итоговый балл умножается на коэффициент:

| Опоздание | Коэффициент |
|---|---:|
| 1 день | 0,75 |
| 2 дня | 0,50 |
| 3 дня | 0,25 |

Ориентир по времени работы — 8–10 часов.

## Что нужно сделать

Исследуйте данные, найдите кластеры и проверьте, как они меняются при преобразовании признаков, сжатии данных и повторном обучении. Объясните различия между кластерами и оцените устойчивость результата.

**Свой вариант посмотрите на Stepik.** Выполняйте только свой вариант.

| Вариант | Данные | Что исследовать |
|---|---|---|
| A · Яндекс Музыка | 5000 пользователей, 8 признаков | Режимы активности и постановки лайков |
| B · Радиосигналы | 5000 кандидатов, 8 признаков | Типы сигналов и необычные наблюдения |
| C · Датчики смартфона | 5000 окон наблюдений, 12 признаков | Режимы движения |
| D · Велопрокат | 352 дня, 24 почасовых значения | Объём аренд и распределение спроса в течение дня |

Все данные находятся в `02_Данные.zip`. Достаточно обычного компьютера или Colab; описание признаков — в конце ноутбука. Готова только загрузка данных. Остальной код напишите самостоятельно; можно использовать материалы занятий.

## Как использовать данные

Вы описываете выданный набор целиком, поэтому для этой работы разделять его на train/test не нужно. Перенос результата на новых пользователей или будущие периоды потребует отдельной проверки.

`object_id` — ID строки, `group_id` — служебная метка для повторной выборки в задании 5. Не включайте их в признаки. Остальные поля из файлов `*_metadata.csv` открывайте только в задании 7, после выбора модели. Не используйте известные классы для подбора числа кластеров, признаков или параметров.

## Загрузка данных

Положите `02_Данные.zip` рядом с ноутбуком; в Colab загрузите его через панель файлов. Укажите свой вариант в строке `VARIANT`. Ячейка загрузит таблицу признаков `data` и служебные метки `groups` для задания 5.

[Скачать данные](https://drive.google.com/file/d/1CG0kt6bm4E1oEJuxdr3HpVEx-4IHhv9Z/view) · [Папка задания](https://drive.google.com/drive/folders/12d4DMQYQbctGG73a0OXTbdAhOXMkb5He)

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import pandas as pd

VARIANT = "A"  # укажите свой вариант из Stepik: A, B, C или D
DATA_FILE = Path("02_Данные.zip")

with ZipFile(DATA_FILE) as archive:
    data = pd.read_csv(archive.open(f"data/{VARIANT}_features.csv"), index_col="object_id")
    groups = pd.read_csv(archive.open(f"data/{VARIANT}_metadata.csv"),
                         usecols=["object_id", "group_id"], dtype=str).set_index("object_id")
data.index = data.index.astype(str)
groups = groups.loc[data.index, "group_id"]
print("Данные:", data.shape)

## 1. Данные и подготовка признаков · 15 баллов

1. Опишите, что означает одна строка данных и для чего могут пригодиться найденные кластеры. До расчётов запишите две гипотезы: о возможных кластерах и о влиянии преобразования признаков.
2. Проверьте типы, пропуски, постоянные столбцы, одинаковые строки и распределения признаков. Покажите хотя бы один график и поясните его. Одинаковые измерения не удаляйте без обоснования.
3. Подготовьте два набора признаков: базовый и изменённый.

**Базовый:** все выданные признаки после `StandardScaler`.

**Изменённый — по вашему варианту:**

| Вариант | Преобразование |
|---|---|
| A | Примените `log1p` к `like_count`, `unique_tracks`, `active_days`, `span_days`, `likes_per_active_day`, `median_gap_hours`. Затем примените `StandardScaler` ко всем 8 признакам. Доли не логарифмируйте. |
| B | Примените `RobustScaler` ко всем 8 признакам. Используйте медиану и межквартильный размах. |
| C | Оставьте 6 признаков со `std()` в названии и примените `StandardScaler`. |
| D | Разделите 24 почасовых числа каждого дня на общее число аренд за этот день. Затем примените `StandardScaler` к полученным столбцам. |

Объясните, какие различия между объектами после преобразования влияют на расстояния сильнее или слабее. Для D сравниваются число аренд и их доли по часам.

Ваши пояснения и выводы.

## 2. Реализация k-means · 15 баллов

Реализуйте k-means на NumPy и проверьте его на небольшом синтетическом примере. Нужны назначение ближайшего центра, обновление центров и расчёт суммы квадратов расстояний до них (`inertia`). При равных расстояниях выбирайте центр с меньшим индексом; пустому кластеру сохраняйте прежний центр.

Покажите, что обновление центров при фиксированных назначенных кластерах не увеличивает `inertia`. Останавливайте цикл, когда центры перестают меняться, но не позднее 100 итераций. Сверьте центры и `inertia` со sklearn при тех же стартовых центрах, `n_init=1` и строгом допуске `tol`, учитывая возможную перестановку номеров кластеров. В следующих заданиях используйте sklearn.

Ваши пояснения и выводы.

## 3. Сравнение и выбор кластеров · 20 баллов

Обучите k-means для обоих наборов признаков из задания 1: `K=2…6`, `random_state=42`, `n_init=20`. Получится 10 вариантов модели.

Для оценки силуэта (`silhouette`) один раз выберите случайные `min(1000, N)` объектов с seed=42. Здесь N — число строк. Используйте одни и те же объекты во всех сравнениях.

Для каждого варианта модели запишите:

- набор признаков и K;
- `inertia` и размеры всех кластеров, включая минимальный размер;
- силуэт на признаках, по которым обучалась модель;
- силуэт с теми же метками кластеров на **базовых стандартизованных признаках**.

Последняя оценка позволяет сравнить разбиения при одинаковом расчёте расстояний. Значения `inertia` для разных наборов признаков напрямую не сравнивайте. Если силуэт не определён, укажите причину вместо числа.

Покажите таблицу результатов и графики `inertia` и силуэта по K. По метрикам, размерам и медианным значениям признаков в кластерах выберите набор признаков и K. Обоснуйте выбор. Если убедительных кластеров нет, покажите, на чём основан этот вывод.

Ваши пояснения и выводы.

## 4. PCA и потеря информации · 20 баллов

Работайте с **базовыми стандартизованными признаками** из задания 1.

1. Реализуйте PCA через NumPy SVD: центрирование, направления, координаты и восстановление данных. Найдите минимальное число компонент q, сохраняющее 90% дисперсии. Покажите накопленную долю дисперсии и ошибку восстановления; сверьте восстановление со sklearn PCA при том же q.
2. При K из задания 3, `random_state=42` и `n_init=20` сравните k-means на всех базовых признаках и на q компонентах. Посчитайте ARI между разбиениями и оба силуэта на базовых стандартизованных признаках.
3. На одних координатах первых двух компонент покажите оба разбиения рядом. Подпишите, какую долю дисперсии сохраняют эти две компоненты.

Объясните, что изменилось после сжатия. Учитывайте, что направления PCA могут отличаться знаком, а номера кластеров — переставляться. ARI сравнивает разбиения независимо от номеров кластеров. Сохранение дисперсии не гарантирует сохранения кластеров.

Ваши пояснения и выводы.

## 5. Устойчивость и описание кластеров · 15 баллов

### Повторное обучение

Зафиксируйте выбранные преобразования и K. Пять раз отберите 80% значений `group_id` случайно, без возвращения, с seed=101…105. Каждый раз берите **все строки** с выбранными значениями:

| Вариант | Что отбирать целиком |
|---|---|
| A | Пользователей: один `group_id` соответствует одному пользователю. |
| B | Строки с кандидатами радиосигнала: у каждой строки свой `group_id`. |
| C | Участников эксперимента со всеми их окнами наблюдений. |
| D | Календарные недели со всеми входящими в них днями. |

Здесь `group_id` задаёт порядок отбора данных, а не номер кластера. В C не выбирайте окна одного человека по отдельности, в D — отдельные дни: соседние наблюдения связаны.

На каждой выбранной части обучите k-means и назначьте кластеры **всем исходным объектам**. Параметры преобразований не переобучайте. Покажите пять ARI относительно исходного выбранного разбиения и оцените устойчивость. Это проверка чувствительности к составу данных; она не измеряет качество на новых объектах и не даёт доверительного интервала. Для B неизвестно, относятся ли разные строки к одному небесному источнику; недельный отбор в D также не доказывает независимость недель.

### Описание результата

Покажите размеры кластеров и медианы исходных признаков. Для двух кластеров приведите минимум два численных различия, дайте понятные названия и поясните ограничения такого описания. Для D постройте медианные почасовые кривые в числе аренд и в долях за день. Для A описывайте действия пользователей, для C учитывайте, что признаки уже нормированы источником.

Ваши пояснения и выводы.

## 6. DBSCAN и необычные объекты · 10 баллов

Исследуйте DBSCAN на базовых стандартизованных признаках при `min_samples=10`. Покажите распределение расстояний до десятого соседа, включая саму точку. Возьмите три eps: квантили 0,50; 0,75; 0,90 этих расстояний.

Для каждого eps укажите число и размеры кластеров без шума, долю шума и долю объектов, попавших в кластеры. Если считаете силуэт, используйте только нешумовые объекты из подвыборки задания 3 и укажите их число. При недопустимом числе кластеров объясните, почему метрика не определена. Не сравнивайте эту оценку напрямую с k-means на всех объектах.

Для одного eps выберите случайно 5 шумовых и 5 нешумовых объектов; при нехватке возьмите все доступные. Покажите ID и признаки. Опишите, какие дополнительные данные нужны эксперту и что он должен проверить или отметить в этих примерах. Шум DBSCAN сам по себе не доказывает ошибку или особую природу объекта; просмотр 10 примеров не оценивает точность детектора.

Ваши пояснения и выводы.

## 7. Итог и дополнительная проверка · 5 баллов

До просмотра отложенных метаданных зафиксируйте в ноутбуке выбранные признаки, преобразования и их параметры, K, seed, `n_init`, eps и ID подвыборки. Покажите назначения кластеров по `object_id`.

После этого можно выполнить дополнительную проверку: для B/C — ARI и таблицу сопоставления с известными классами, для D — распределение типов дней по кластерам. Для A внешних классов нет. Проверка необязательна; после неё модель не меняйте. Высокое совпадение с классами дополнительных баллов не даёт.

Напишите вывод на 250–400 слов: вопрос, выбранное решение, ключевые числа, устойчивость, эффект PCA, трактовка шума, ограничения и следующая проверка. Ответьте на индивидуальный вопрос из Stepik. Укажите, что потребуется для работы с новыми объектами или будущими периодами.

Ваши пояснения и выводы.

## Оценивание

| Блок | Баллы | За что начисляются |
|---|---:|---|
| Данные и подготовка признаков | 15 | Постановка и гипотезы 5; аудит 5; два набора признаков и их обоснование 5 |
| Реализация k-means | 15 | Назначение 4; центры и пустые кластеры 4; целевая функция и сверка 7 |
| Сравнение и выбор кластеров | 20 | Таблица 10 моделей 8; сопоставимые оценки 6; обоснованный выбор 6 |
| PCA | 20 | SVD и проверка восстановления 8; дисперсия и q 5; сравнение разбиений и графики 7 |
| Устойчивость и описание | 15 | Пять повторов с правильным отбором и интерпретацией 8; описание кластеров 7 |
| DBSCAN и проверка примеров | 10 | Три запуска, доля нешумовых объектов и корректные метрики 6; план проверки 4 |
| Воспроизводимость и итог | 5 | Полный запуск, фиксация выбора и обоснованный вывод |
| **Всего** | **100** | |

Баллы начисляются за выполненные пункты. Если известные классы использованы для подбора модели, соответствующие пункты выбора и выводов не засчитываются; остальные части оцениваются отдельно. Низкая метрика или обоснованный вывод об отсутствии пригодных кластеров сами по себе не снижают оценку.

## Что сдавать

Один выполненный ноутбук с кодом, таблицами, графиками и выводами. Укажите версии Python и библиотек, устройство и время полного запуска. Перед сдачей перезапустите среду и выполните ячейки по порядку.

Дополнительный отчёт, презентация, архив исходных данных и отдельные таблицы не нужны. Достаточно CPU и библиотек NumPy, pandas, matplotlib, scikit-learn и SciPy.

# Справочник данных

В `*_features.csv` находятся признаки и `object_id`. В `*_metadata.csv` — `group_id` для задания 5 и отложенные сведения для задания 7. ID не используйте как признаки.

## A · Яндекс Музыка / Yambda

Одна строка — пользователь. Выданы 5000 пользователей с минимум 20 событиями постановки лайка из [Yambda-50M](https://huggingface.co/datasets/yandex/yambda). Пользователи без лайков и с редкими лайками в эту выборку не входят.

| Признак | Что означает |
|---|---|
| `like_count` | Число событий постановки лайка |
| `unique_tracks` | Число разных треков в этих событиях |
| `active_days` | Число суток, в которые поставлен хотя бы один лайк |
| `span_days` | Время между первым и последним лайком, в сутках |
| `organic_share` | Доля лайков при самостоятельном выборе трека, без перехода из рекомендаций; от 0 до 1 |
| `peak_day_share` | Максимальная доля лайков пользователя, поставленная за одни сутки |
| `likes_per_active_day` | Число лайков, делённое на `active_days` |
| `median_gap_hours` | Медиана времени между последовательными лайками, в часах; нули допустимы |

Сутки отсчитываются от условного начала набора. Реальные даты, часовой пояс, названия и жанры треков не раскрыты. Считаются события лайков без вычитания отмен: это не размер текущего избранного и не число прослушиваний. `organic_share` описывает канал действия, но не доказывает влияние рекомендаций на поведение. Правильных внешних классов для этого варианта нет. `group_id` — ID пользователя.

## B · Радиосигналы HTRU2

Одна строка — кандидат радиосигнала. Выданы 5000 строк из [HTRU2](https://archive.ics.uci.edu/dataset/372/htru2).

- `profile_*` описывают форму усреднённого импульса.
- `dm_*` описывают кривую отношения сигнал/шум при разных настройках обработки сигнала.
- Для каждого описания даны среднее (`mean`), стандартное отклонение (`std`), асимметрия (`skewness`) и эксцесс (`excess_kurtosis`). Среднее и std — в шкалах исходного сигнала, асимметрия и эксцесс безразмерны.

`group_id` совпадает с ID строки: неизвестно, могут ли разные строки относиться к одному небесному источнику. Отложенная метка `audit_label`: 1 — подтверждённый пульсар, 0 — отрицательный пример. Полных наблюдений для экспертной проверки в таблице нет.

## C · Датчики смартфона / UCI HAR

Одна строка — окно измерений длительностью 2,56 секунды. Выданы 5000 окон из [UCI HAR](https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones).

12 признаков — средние (`mean()`) и стандартные отклонения (`std()`) по осям X/Y/Z:

- `tBodyAcc-*` — характеристики ускорения тела;
- `tBodyGyro-*` — характеристики угловой скорости.

Источник уже нормировал значения примерно в диапазон [-1; 1]. Их нельзя читать как физические ускорения или угловые скорости. Окна одного участника перекрываются на 50%, поэтому при повторном отборе сохраняйте все его окна вместе. `group_id` — участник эксперимента.

Отложенная метка `audit_label`: 1 — ходьба, 2 — подъём по лестнице, 3 — спуск, 4 — сидение, 5 — стояние, 6 — положение лёжа. Эти классы не задают число кластеров. Официальные train/test объединены для описания выданного набора; качество на новых людях здесь не проверяется.

## D · Велопрокат Сеула

Одна строка — день работы проката. Выданы 352 дня из [Seoul Bike Sharing Demand](https://archive.ics.uci.edu/dataset/560/seoul+bike+sharing+demand).

`hour_00`…`hour_23` — число аренд за каждый час этого дня. Оставлены полные дни, когда прокат работал все 24 часа; 13 дней с часами неработающего сервиса исключены. `group_id` — календарная неделя.

В исходных числах сравнивается объём спроса. После деления на сумму за день сравнивается распределение аренд по часам; последующая стандартизация также меняет вес каждого часа. Сезон, праздничность и тип дня находятся в отложенных метаданных. Они помогают описать кластеры, но не доказывают причины изменения спроса.

## Источники и подготовка

Yambda: Copyright 2025 YANDEX LLC, Apache 2.0; [статья авторов](https://arxiv.org/abs/2505.22238). Источники трёх остальных наборов — UCI, CC BY 4.0. Данные подготовлены преподавателем: выбраны строки и признаки, для A рассчитаны показатели по пользователям, для D почасовые записи объединены по дням. Ссылки на источники, точные правила отбора и контрольные суммы сохранены в `datasets.json` внутри архива; лицензия Yambda также включена в архив.